In [1]:
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# LOad the dataset
diamonds = sns.load_dataset('diamonds')


diamonds.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [3]:
X = diamonds.drop('cut', axis=1)
y= diamonds['cut']

In [4]:
# 2. Split the data

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [5]:
# define categorical and numerical columns
categorical_features = X.select_dtypes(include=['object']).columns.to_list()

numerical_features = X.select_dtypes(include=['float64','int64']).columns.tolist()

print(f"Category cols: {categorical_features}")
print(f"Numerical cols: {numerical_features}")

Category cols: []
Numerical cols: ['carat', 'depth', 'table', 'price', 'x', 'y', 'z']


In [6]:
# preprocess columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

In [16]:
#Gradient boosting classifier pipeline

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classfier", GradientBoostingClassifier(
            random_state=42, 
            n_estimators=1000, 
            learning_rate=0.1, 
            subsample=0.8,
            min_samples_leaf=1,
            min_samples_split=2,
            min_weight_fraction_leaf=0.0,
            max_depth=5,
            min_impurity_decrease=0.0,
            max_features=None,
            verbose=0,
            max_leaf_nodes=None,
            warm_start=False,
            validation_fraction=0.1, 
            n_iter_no_change=None, 
            tol=0.0001, 
            ccp_alpha=0.0
            ))
    ]
)

In [13]:
#CV and training

# Perform 5 fold cross-validation
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)

# Fit the modal to train data

pipeline.fit(X_train, y_train)

# predict the test set

y_pred = pipeline.predict(X_test)

# generate report 

report = classification_report(y_test, y_pred)






In [14]:
# Report the final results

print(f"Mean Cross-Validation Accuracy: {cv_scores.mean():.4f}")

print(f"\nClassification report: \n{report}")

Mean Cross-Validation Accuracy: 0.7879

Classification report: 
              precision    recall  f1-score   support

        Fair       0.89      0.92      0.90       335
        Good       0.79      0.73      0.76      1004
       Ideal       0.83      0.90      0.87      4292
     Premium       0.84      0.82      0.83      2775
   Very Good       0.70      0.63      0.66      2382

    accuracy                           0.80     10788
   macro avg       0.81      0.80      0.80     10788
weighted avg       0.80      0.80      0.80     10788

